In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [2]:
random_int8 = torch.randint(-128, 127, (32, 16)).to(torch.int8)
random_hs = torch.randn((1, 16), dtype=torch.bfloat16)
scales = torch.randn((1, 32), dtype=torch.bfloat16)
bias = torch.randn((1, 32), dtype=torch.bfloat16)

In [3]:
F.linear(random_hs, random_int8.to(random_hs.dtype))

tensor([[  -7.6875, -478.0000,  396.0000,  131.0000,  141.0000, -502.0000,
          -45.0000,  490.0000,  193.0000, -454.0000,  144.0000, -256.0000,
          452.0000, -760.0000, -114.5000, -157.0000,  560.0000,  556.0000,
          120.0000, -544.0000, -812.0000,  764.0000, -144.0000, -472.0000,
           76.5000,  256.0000, -202.0000,  512.0000, -148.0000, -636.0000,
          104.0000,   69.0000]], dtype=torch.bfloat16)

In [4]:
F.linear(random_hs, random_int8.to(random_hs.dtype)) * scales

tensor([[   8.4375,  248.0000, -251.0000,  -37.2500,  -68.5000,  238.0000,
          -26.0000,  354.0000, -173.0000,  676.0000,  138.0000, -328.0000,
          153.0000, 1672.0000, -117.0000,  191.0000,  147.0000,  604.0000,
          -32.5000, 1144.0000,   47.0000,   90.5000,   15.0625, -296.0000,
            7.7188, -358.0000,  302.0000, -384.0000,   72.0000, -736.0000,
          -59.7500,   41.5000]], dtype=torch.bfloat16)

In [5]:
(F.linear(random_hs, random_int8.to(random_hs.dtype)) * scales) + bias

tensor([[   9.8750,  249.0000, -251.0000,  -37.0000,  -68.0000,  238.0000,
          -25.3750,  354.0000, -175.0000,  676.0000,  138.0000, -330.0000,
          153.0000, 1672.0000, -117.5000,  189.0000,  145.0000,  604.0000,
          -32.7500, 1144.0000,   46.0000,   89.5000,   15.1250, -298.0000,
            6.4688, -356.0000,  302.0000, -382.0000,   71.5000, -736.0000,
          -61.0000,   40.5000]], dtype=torch.bfloat16)

In [6]:
def w8_a16_forward(weight, input, scales, bias=None):
    
    casted_weights = weight.to(input.dtype)
    output = F.linear(input, casted_weights) * scales
    
    if bias is not None:
        output = output + bias
      
    return output

In [ ]:
print("With bias:\n\n", 
      w8_a16_forward(random_int8, random_hs, scales, bias))

print("\nWithout bias:\n\n", 
      w8_a16_forward(random_int8, random_hs, scales))

In [ ]:
### running this will result in an error
class W8A16LinearLayer(nn.Module):
    def __init__(self, in_features, out_features, 
                 bias=True, dtype=torch.float32):
        super().__init__()
        
        self.int8_weights = nn.Parameter(torch.Tensor([0, 1]
                                     ).to(dtype=torch.int8))

try:
    
    W8A16LinearLayer(1, 1)
    
except Exception as error:
    print("\033[91m", type(error).__name__, ": ", error, "\033[0m")

In [ ]:
class W8A16LinearLayer(nn.Module):
    def __init__(self, in_features, out_features, 
                 bias=True, dtype=torch.float32):
        super().__init__()
        
        
        self.register_buffer(
            "int8_weights",
            torch.randint(
                -128, 127, (out_features, in_features), dtype=torch.int8
            )
        )
        
        self.register_buffer("scales", 
                             torch.randn((out_features), dtype=dtype))
        
        if bias:
            self.register_buffer("bias", 
                                 torch.randn((1, out_features), 
                                             dtype=dtype))
        
        else:
            self.bias = None

In [ ]:
dummy_instance = W8A16LinearLayer(16, 32)

In [ ]:
print(dummy_instance.int8_weights.shape)
print(dummy_instance.scales.shape)

In [ ]:
class W8A16LinearLayer(nn.Module):
    def __init__(self, in_features, out_features, 
                 bias=True, dtype=torch.float32):
        super().__init__()
        
        
        self.register_buffer(
            "int8_weights",
            torch.randint(
                -128, 127, (out_features, in_features), dtype=torch.int8
            )
        )
        
        self.register_buffer("scales", 
                             torch.randn((out_features), dtype=dtype))
        
        if bias:
            self.register_buffer("bias", 
                                 torch.randn((1, out_features), 
                                             dtype=dtype))
        
        else:
            self.bias = None

    def forward(self, input):
        return w8_a16_forward(self.int8_weights, 
                              input, self.scales, self.bias)

In [ ]:
module = W8A16LinearLayer(16, 32)
dummy_hidden_states = torch.randn(1, 6, 16)

In [ ]:
module(dummy_hidden_states).shape

In [ ]:
module(dummy_hidden_states).dtype

In [ ]:
class W8A16LinearLayer(nn.Module):
    def __init__(self, in_features, out_features, 
                 bias=True, dtype=torch.float32):
        super().__init__()
        
        
        self.register_buffer(
            "int8_weights",
            torch.randint(
                -128, 127, (out_features, in_features), dtype=torch.int8
            )
        )
        
        self.register_buffer("scales", 
                             torch.randn((out_features), dtype=dtype))
        
        if bias:
            self.register_buffer("bias", 
                                 torch.randn((1, out_features), 
                                             dtype=dtype))
        
        else:
            self.bias = None

    def quantize(self, weights):
        w_fp32 = weights.clone().to(torch.float32)

        scales = w_fp32.abs().max(dim=-1).values / 127
        scales = scales.to(weights.dtype)

        int8_weights = torch.round(weights
                        /scales.unsqueeze(1)).to(torch.int8)

        self.int8_weights = int8_weights
        self.scales = scales
    
    def forward(self, input):
        return w8_a16_forward(self.int8_weights, 
                              input, self.scales, self.bias)      

In [ ]:
module = W8A16LinearLayer(4, 8)
print("Weights before:\n" , module.int8_weights)
random_matrix = torch.randn((4, 8), dtype=torch.bfloat16)
module.quantize(random_matrix)
print("Weights After:\n" , module.int8_weights)
module.scales
module.scales.shape
module.int8_weights.shape
### dequantized weights
module.int8_weights * module.scales.unsqueeze(1)
### original weights
random_matrix
(random_matrix - module.int8_weights 
 * module.scales.unsqueeze(1)).abs().mean()